In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    PolynomialFeatures,
)
from sklearn.linear_model import LinearRegression

In [2]:
dataset_path = Path(
    "../data/processed/airfoil_dataset_final.csv"
)

data = pd.read_csv(dataset_path)

features = [
    "max_camber",
    "camber_position",
    "thickness",
    "reynolds",
    "alpha",
]

X = data[features]


cl_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "poly",
        PolynomialFeatures(
            degree=3,
            include_bias=False,
        ),
    ),
    ("regression", LinearRegression()),
])


cd_model = Pipeline([
    ("scaler", StandardScaler()),
    (
        "poly",
        PolynomialFeatures(
            degree=3,
            include_bias=False,
        ),
    ),
    ("regression", LinearRegression()),
])


cl_model.fit(
    X,
    data["CL"],
)

cd_model.fit(
    X,
    data["CD"],
)

print("Final models trained.")

Final models trained.


In [3]:
n_cases = 10000

rng = np.random.default_rng(
    seed=42
)

performance_inputs = pd.DataFrame({
    "max_camber":
        rng.uniform(
            0.0,
            0.04,
            n_cases,
        ),

    "camber_position":
        rng.uniform(
            0.2,
            0.6,
            n_cases,
        ),

    "thickness":
        rng.uniform(
            0.09,
            0.15,
            n_cases,
        ),

    "reynolds":
        rng.uniform(
            100000,
            500000,
            n_cases,
        ),

    "alpha":
        rng.uniform(
            -6,
            12,
            n_cases,
        ),
})

In [4]:
prediction_times = []

for _ in range(20):

    start = time.perf_counter()

    CL_predictions = cl_model.predict(
        performance_inputs
    )

    CD_predictions = cd_model.predict(
        performance_inputs
    )

    end = time.perf_counter()

    prediction_times.append(
        end - start
    )


median_prediction_time = np.median(
    prediction_times
)

time_per_prediction = (
    median_prediction_time
    / n_cases
)


print(
    "SURROGATE PERFORMANCE"
)

print(
    "---------------------"
)

print(
    f"Cases predicted: {n_cases}"
)

print(
    f"Median total time: "
    f"{median_prediction_time:.6f} s"
)

print(
    f"Time per operating point: "
    f"{time_per_prediction:.8f} s"
)

print(
    f"Predictions per second: "
    f"{1 / time_per_prediction:.0f}"
)

SURROGATE PERFORMANCE
---------------------
Cases predicted: 10000
Median total time: 0.012914 s
Time per operating point: 0.00000129 s
Predictions per second: 774356


In [5]:
import sys

sys.path.append(
    "../src"
)

from xfoil_runner import run_xfoil

In [6]:
xfoil_times = []

for i in range(3):

    print(
        f"XFOIL timing run {i + 1}/3"
    )

    start = time.perf_counter()

    run_xfoil(
        naca="2412",
        reynolds=300000,
        alpha_start=-6,
        alpha_end=12,
        alpha_step=1,
    )

    end = time.perf_counter()

    xfoil_times.append(
        end - start
    )


median_xfoil_time = np.median(
    xfoil_times
)

xfoil_time_per_point = (
    median_xfoil_time
    / 19
)


print()
print("XFOIL PERFORMANCE")
print("-----------------")

print(
    f"Median 19-point sweep: "
    f"{median_xfoil_time:.4f} s"
)

print(
    f"Approximate time per point: "
    f"{xfoil_time_per_point:.4f} s"
)

XFOIL timing run 1/3

--------------------------------------
Running XFOIL
--------------------------------------
Airfoil: NACA 2412
Reynolds number: 300000
Angle range: -6 to 12 degrees

XFOIL run completed successfully.
Results saved to:
C:\Users\levi3\OneDrive\Documents\GitHub\MMA3001-Airfoil-Surrogate\data\raw\naca2412_re300000_auto.txt
XFOIL timing run 2/3

--------------------------------------
Running XFOIL
--------------------------------------
Airfoil: NACA 2412
Reynolds number: 300000
Angle range: -6 to 12 degrees

XFOIL run completed successfully.
Results saved to:
C:\Users\levi3\OneDrive\Documents\GitHub\MMA3001-Airfoil-Surrogate\data\raw\naca2412_re300000_auto.txt
XFOIL timing run 3/3

--------------------------------------
Running XFOIL
--------------------------------------
Airfoil: NACA 2412
Reynolds number: 300000
Angle range: -6 to 12 degrees

XFOIL run completed successfully.
Results saved to:
C:\Users\levi3\OneDrive\Documents\GitHub\MMA3001-Airfoil-Surrogate\data\ra

In [7]:
speedup = (
    xfoil_time_per_point
    / time_per_prediction
)

print(
    f"Approximate surrogate speedup: "
    f"{speedup:,.0f}x"
)

Approximate surrogate speedup: 76,057x
